<a href="https://colab.research.google.com/github/dndivaka/report-app/blob/main/Product_Dashboard_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
#  SCHOOL USAGE DASHBOARD GENERATOR — GOOGLE COLAB
# ================================================================
#
#  HOW TO USE:
#  1. Open Google Colab  →  colab.research.google.com
#  2. Create a new notebook  (File → New notebook)
#  3. Click the first cell, paste THIS ENTIRE FILE
#  4. Press  Shift + Enter  to run
#  5. An upload button appears — click it and upload your .xlsx
#  6. Enter working days and eligible teachers when asked
#  7. Excel + HTML files download automatically when done
#
# ================================================================

# ── Step 0: Install dependencies ────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "--quiet", "pandas", "openpyxl"])

# ── Step 1: Imports ──────────────────────────────────────────────
import os, re, io, tempfile
from datetime import datetime

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference
from openpyxl.utils import get_column_letter

print("✅ Libraries loaded")

# ================================================================
#  CONSTANTS — Subject remapping & recommended minutes
# ================================================================

ENGLISH_MAP = {
    'English Language', 'English Literature', 'English - Literature Reader',
    'English Writing and Comprehension', 'English Reading', 'English',
    'English Reader', 'English Conversation', 'English Grammar (Graded)',
    'English I', 'English II',
}
MATH_MAP    = {'Accountancy', 'Statistics'}
HINDI_MAP   = {'Hindi Language', 'Hindi Literature', 'Hindi', 'Hindi Reader',
               'Hindi Grammar (Graded)', 'Hindi I', 'Hindi II'}
SCIENCE_MAP = {
    'Environmental Science', 'Computer Science', 'Physics', 'Biology',
    'Chemistry', 'Psychology', 'Science Practicals', 'Artificial Intelligence', 'Science',
}
SOCIAL_MAP  = {
    'Economics', 'Business Studies', 'Geography', 'History',
    'Political Science', 'Social Studies', 'Social Science',
}

# Subjects that stay as their original name for Class 11 & 12
# (not merged into Science / Mathematics / Social Studies groups)
SENIOR_INDEPENDENT_SUBJECTS = {
    'Accountancy', 'Statistics', 'Physics', 'Biology', 'Chemistry',
    'Psychology', 'Computer Science', 'Artificial Intelligence',
    'Economics', 'Business Studies', 'Geography', 'History', 'Political Science',
}
SENIOR_GRADES = {'Class 11', 'Class 12'}

# ════════════════════════════════════════════════════════════
#  ELIGIBLE SUBJECTS
#  Any teacher whose "Subjects taught" contains at least one
#  subject from this set is counted as an eligible teacher.
#  ADD NEW SUBJECTS HERE as they become eligible.
# ════════════════════════════════════════════════════════════
ELIGIBLE_SUBJECTS = {
    'Art & Craft', 'Computer Science', 'English', 'English Language',
    'English Literature', 'English Reading', 'English - Literature Reader',
    'English Reader', 'English Writing and Comprehension',
    'Environmental Science', 'General Knowledge', 'Gujarati',
    'Hindi', 'Hindi Language', 'Hindi Literature', 'Hindi Reader',
    'Mathematics', 'Value Education', 'Sanskrit Literature',
    'Sanskrit Grammar', 'Science', 'Social Studies', 'Social Science',
    'Artificial Intelligence', 'Geography', 'History', 'Political Science',
    'Biology', 'Biology Practicals', 'Chemistry', 'Chemistry Practicals',
    'Economics', 'Physics', 'Physics Practicals', 'Science Practicals',
    'Accountancy', 'Business Studies', 'Informatics Practices',
    'Physical Education', 'Psychology', 'Sociology', 'Statistics',
    'Conservation and Civic Sense', 'Inside Industry', 'Nobel Prizes',
    'Adolescent Education', 'Marathi', 'Commerce',
}



_EARLY_REC = {'English': 360,  'Science': 270,  'Hindi': 180,  'Mathematics': 360}
_C1_5_REC  = {'English': 1200, 'Science': 1500, 'Hindi': 600,  'Mathematics': 1800}
_C6_12_REC = {'English': 1200, 'Science': 1500, 'Hindi': 600,  'Mathematics': 1800,
              'Social Studies': 1500}

RECOMMENDED_MINS = {
    'Nursery':                   _EARLY_REC,
    'Junior KG':                 _EARLY_REC,
    'Senior KG':                 _EARLY_REC,
    'Early Childhood Education': _EARLY_REC,
    'Primary':                   _EARLY_REC,
    **{f'Class {i}': _C1_5_REC  for i in range(1, 6)},
    **{f'Class {i}': _C6_12_REC for i in range(6, 13)},
}

GRADE_ORDER = [
    'Nursery', 'Junior KG', 'Senior KG', 'Early Childhood Education', 'Primary',
    *[f'Class {i}' for i in range(1, 13)],
]

# ================================================================
#  STYLE HELPERS
# ================================================================

HDR = "1F4E79"; ACC = "2E86C1"; ALT = "D6E4F0"
GRN = "1E8449"; ORG = "D35400"

def _thin():
    t = Side(style='thin', color='AAAAAA')
    return Border(left=t, right=t, top=t, bottom=t)

def _med():
    m = Side(style='medium', color='1F4E79')
    return Border(left=m, right=m, top=m, bottom=m)

def _fill(c):    return PatternFill("solid", fgColor=c)
def _font(bold=True, color="FFFFFF", size=11, italic=False):
    return Font(name="Calibri", bold=bold, color=color, size=size, italic=italic)
def _center():   return Alignment(horizontal='center', vertical='center', wrap_text=True)
def _left():     return Alignment(horizontal='left',   vertical='center', wrap_text=False)
def _style(cell, bg=None, font=None, align=None, border=None):
    if bg:     cell.fill      = _fill(bg)
    if font:   cell.font      = font
    if align:  cell.alignment = align
    if border: cell.border    = border

def _hdr_row(ws, row, headers, widths, bg=ACC, height=26):
    for col, (h, w) in enumerate(zip(headers, widths), 1):
        ws.column_dimensions[get_column_letter(col)].width = w
        c = ws.cell(row, col, h)
        _style(c, bg=bg, font=_font(size=10), align=_center(), border=_thin())
    ws.row_dimensions[row].height = height

def _title(ws, text, cols, bg=HDR, size=13):
    ws.merge_cells(f"A1:{get_column_letter(cols)}1")
    ws["A1"] = text
    _style(ws["A1"], bg=bg, font=_font(size=size), align=_center())
    ws.row_dimensions[1].height = 34

def _cls_color(grade):
    early = {'Nursery','Junior KG','Senior KG','Early Childhood Education','Primary'}
    if grade in early: return ('F4D03F','7D6608')
    m = re.search(r'\d+', grade)
    n = int(m.group()) if m else 0
    if 1  <= n <= 5:  return ('2E86C1','FFFFFF')
    if 6  <= n <= 10: return ('1E8449','FFFFFF')
    return ('884EA0','FFFFFF')

def _row(ws, er, vals, bg, lefts=()):
    ws.row_dimensions[er].height = 20
    for col, val in enumerate(vals, 1):
        c = ws.cell(er, col, val)
        c.fill = _fill(bg); c.border = _thin()
        c.alignment = _left() if col in lefts else _center()

# ================================================================
#  PROCESSING FUNCTIONS
# ================================================================

def remap_subject(s, grade=None):
    """
    Remap subjectTitle to a consolidated group name.
    For Class 11 & 12: subjects in SENIOR_INDEPENDENT_SUBJECTS
    keep their original name (not merged into any group).
    For all other grades: full merging applies.
    """
    if pd.isna(s): return s
    s = str(s).strip()

    # Class 11 & 12 — keep these subjects separate
    if str(grade) in SENIOR_GRADES and s in SENIOR_INDEPENDENT_SUBJECTS:
        return s

    # All grades — standard remapping
    if s in ENGLISH_MAP: return 'English'
    if s in MATH_MAP:    return 'Mathematics'
    if s in HINDI_MAP:   return 'Hindi'
    if s in SCIENCE_MAP: return 'Science'
    if s in SOCIAL_MAP:  return 'Social Studies'
    return s


def process_user_summary(xl, working_days):
    s_pfx = '   ' if 'School_Dashboard_Colab.py' == 'School_Dashboard_Colab.py' else '     '
    print(f"\n{s_pfx}Processing Teacher Metrics (UserSummary)...")
    df   = xl.parse('UserSummary')
    mask = df['Name'].str.contains('Support Admin', case=False, na=False)
    removed = df[mask]['Name'].tolist()
    df   = df[~mask].reset_index(drop=True)
    print(f"{s_pfx} Removed Support Admin: {removed}")
    print(f"{s_pfx} Total teachers (excl Admin): {len(df)}")
    total = len(df)

    # Auto-compute eligible count from Subjects taught column
    def _is_eligible(subj_str):
        if pd.isna(subj_str): return False
        return any(s.strip() in ELIGIBLE_SUBJECTS for s in str(subj_str).split(','))

    df = df.copy()
    df['Is Eligible'] = df['Subjects taught'].apply(_is_eligible)
    eligible = int(df['Is Eligible'].sum())
    not_elig = df[~df['Is Eligible']]['Name'].tolist()
    print(f"{s_pfx} Eligible teachers (auto from Subjects taught): {eligible}")
    if not_elig:
        print(f"{s_pfx} Not eligible (subject not in list): {not_elig}")

    # Teachers with total monthly session > 30 mins
    above_30     = int((df['Session Duration (mins)'] > 30).sum())
    pct_above_30 = round(above_30 / total * 100, 2) if total else 0
    print(f"{s_pfx} Session > 30 mins total: {above_30}/{total} ({pct_above_30}%)")

    # Avg daily = Session Duration / working days
    df['Avg Daily (mins)'] = (df['Session Duration (mins)'] / working_days).round(2)
    print(f"{s_pfx} Avg daily duration computed over {working_days} working days")

    # Teachers avg daily >= 30 mins ÷ eligible count
    avg_30     = int((df['Avg Daily (mins)'] >= 30).sum())
    pct_avg_30 = round(avg_30 / eligible * 100, 2) if eligible else 0
    print(f"{s_pfx} Avg daily >= 30 mins/day: {avg_30}/{eligible} eligible ({pct_avg_30}%)")

    return df, {'total': total, 'eligible': eligible,
                'above_30': above_30, 'pct_above_30': pct_above_30,
                'avg_30': avg_30,     'pct_avg_30': pct_avg_30}


def process_terminal_summary(xl, working_days):
    print("\n🏫 Processing Classroom Metrics…")
    df    = xl.parse('TerminalSummary').dropna(subset=['Classroom accessed']).reset_index(drop=True)
    total = len(df)
    print(f"   • Total classrooms: {total}")
    above_30     = int((df['Duration(mins)'] > 30).sum())
    pct_above_30 = round(above_30 / total * 100, 2)
    print(f"   • Duration > 30 mins: {above_30}/{total} ({pct_above_30}%)")
    df = df.copy()
    df['Avg Daily (mins)'] = (df['Duration(mins)'] / working_days).round(2)
    above_40     = int((df['Avg Daily (mins)'] >= 40).sum())
    pct_above_40 = round(above_40 / total * 100, 2)
    print(f"   • Avg ≥ 40 mins/day: {above_40}/{total} ({pct_above_40}%)")
    return df, {'total': total,
                'above_30': above_30, 'pct_above_30': pct_above_30,
                'above_40': above_40, 'pct_above_40': pct_above_40}


def process_detailed_usage(xl):
    print("\n📋 Processing Detailed Usage…")
    df  = xl.parse('DetailedUsage')
    raw = len(df)
    print(f"   • Raw rows: {raw}")
    df  = df.drop_duplicates(subset=['Name','Date','startTime','endTime'],
                              keep='first').reset_index(drop=True)
    print(f"   • After dedup: {len(df)}  (removed {raw - len(df)})")
    df['startTime_dt']    = pd.to_datetime(df['startTime'], errors='coerce')
    df['endTime_dt']      = pd.to_datetime(df['endTime'],   errors='coerce')
    df['Duration (mins)'] = ((df['endTime_dt'] - df['startTime_dt'])
                              .dt.total_seconds() / 60).round(2)
    print(f"   • Duration computed for {int(df['Duration (mins)'].notna().sum())} rows")
    df['subjectTitle'] = df.apply(lambda row: remap_subject(row['subjectTitle'], row.get('gradeTitle')), axis=1)
    subjects = sorted(df['subjectTitle'].dropna().unique())
    print(f"   • Subject groups: {subjects}")
    return df.drop(columns=['startTime_dt','endTime_dt'])



# ─────────────────────────────────────────────
# RECOMMENDED USAGE MINUTES — FULL TABLE
# ─────────────────────────────────────────────
_EARLY_REC  = {'English':360,  'Science':270,  'Hindi':180,  'Mathematics':360}
_C1_5_REC   = {'English':1200, 'Science':1500, 'Hindi':600,  'Mathematics':1800,
               'General Knowledge':300, 'Art & Craft':300, 'Gujarati':300}
_C6_10_REC  = {'English':1200, 'Science':1500, 'Hindi':600,  'Mathematics':1800,
               'Social Studies':1500, 'Art & Craft':300, 'Gujarati':300,
               'Sanskrit Literature':300}
_C11_12_REC = {'English':1200, 'Science':1500, 'Hindi':600,  'Mathematics':1800,
               'Social Studies':1500,
               'Accountancy':600, 'Biology':600, 'Business Studies':600,
               'Chemistry':600, 'Computer Science':600, 'Economics':600,
               'History':600, 'Physical Education':300, 'Physics':600,
               'Political Science':600, 'Psychology':600}

RECOMMENDED_MINS = {
    'Nursery':                   _EARLY_REC,
    'Junior KG':                 _EARLY_REC,
    'Senior KG':                 _EARLY_REC,
    'Early Childhood Education': _EARLY_REC,
    'Primary':                   _EARLY_REC,
    **{f'Class {i}': _C1_5_REC   for i in range(1, 6)},
    **{f'Class {i}': _C6_10_REC  for i in range(6, 11)},
    **{f'Class {i}': _C11_12_REC for i in range(11, 13)},
}

def get_recommended(subject, grade):
    """Return recommended minutes for subject+grade, or '—' if none defined.
    Note: Social Studies is excluded from Early Childhood Education (data error).
    """
    if grade == 'Early Childhood Education' and subject == 'Social Studies':
        return '—'
    return RECOMMENDED_MINS.get(str(grade), {}).get(subject, '—')

def build_class_tables(det_df):
    """
    Build subject-wise tables where each subject shows all its grades.
    Structure: Subject → [Grade, Recommended (mins), Usage (mins)]
    """
    print("  ──────────────────────────────────────────────────────")
    print("  📚 Building subject-wise grade usage tables…")

    valid = det_df.dropna(subset=['subjectTitle', 'gradeTitle', 'Duration (mins)'])

    # Group by subject → grade
    sg = valid.groupby(['subjectTitle', 'gradeTitle'], as_index=False)['Duration (mins)'].sum().round(2)
    sg.columns = ['Subject Title', 'Grade Title', 'Usage (mins)']

    # Sort grades in logical order
    GRADE_ORDER_LOCAL = [
        'Nursery', 'Junior KG', 'Senior KG', 'Early Childhood Education', 'Primary',
        *[f'Class {i}' for i in range(1, 13)]
    ]
    sg['_grade_sort'] = sg['Grade Title'].apply(
        lambda g: GRADE_ORDER_LOCAL.index(g) if g in GRADE_ORDER_LOCAL else 99
    )
    sg = sg.sort_values(['Subject Title', '_grade_sort']).drop(columns='_grade_sort').reset_index(drop=True)

    # Build dict keyed by subject
    subject_tables = {}
    for subject in sorted(sg['Subject Title'].unique()):
        tbl = sg[sg['Subject Title'] == subject][['Grade Title', 'Usage (mins)']].reset_index(drop=True)
        tbl.insert(1, 'Recommended (mins)',
                   tbl['Grade Title'].apply(lambda g: get_recommended(subject, g)))
        subject_tables[subject] = tbl
        total = round(tbl['Usage (mins)'].sum(), 1)
        print(f"     • {subject:<35} {len(tbl)} grades   total: {total} mins")

    return subject_tables



# ─────────────────────────────────────────────
# DAILY MEDIA / PRINTABLES / WIDGETS SHEET
# ─────────────────────────────────────────────
def process_dashboard_daily(xl):
    """
    Read DashBoard sheet, filter out Saturdays & Sundays,
    return clean DataFrame with Date, Media played,
    Printables used, Widgets Used.
    """
    import numpy as np
    try:
        df = xl.parse('DashBoard', header=6)
    except Exception:
        df = xl.parse('DashBoard', header=None, skiprows=6)
    df.columns = [str(c).strip() for c in df.columns]

    rows = []
    for _, row in df.iterrows():
        date_val = row.get('Dates', None)
        if date_val is None or (isinstance(date_val, float) and np.isnan(date_val)):
            continue
        if str(date_val).strip() in ['no data', 'nan', '']:
            continue
        try:
            dt = pd.to_datetime(date_val)
        except Exception:
            continue
        if dt.weekday() >= 5:   # Skip Saturday (5) and Sunday (6)
            continue
        def safe(v):
            return 0 if (v is None or (isinstance(v, float) and np.isnan(v))) else int(v)
        rows.append({
            'Date':             dt.strftime('%d-%b-%Y'),
            'Day':              dt.strftime('%A'),
            'Media played':     safe(row.get('Media played', 0)),
            'Printables used':  safe(row.get('Printables', 0)),
            'Widgets Used':     safe(row.get('Widgets created', 0)),
        })
    result = pd.DataFrame(rows)
    print(f"     • Daily sheet: {len(result)} weekdays (Sat/Sun excluded)")
    return result


def build_sheet_daily(wb, daily_df):
    """Add Daily Media/Printables/Widgets sheet to workbook."""
    ws = wb.create_sheet("Daily Usage")
    ws.sheet_view.showGridLines = False

    # Column widths
    for col, w in zip('ABCDE', [18, 14, 18, 18, 16]):
        ws.column_dimensions[col].width = w

    # Title
    ws.merge_cells("A1:E1")
    ws["A1"] = "Daily Media, Printables & Widgets Usage"
    _style(ws["A1"], bg=HDR, font=_font(size=13), align=_center())
    ws.row_dimensions[1].height = 34

    # Sub-info
    ws.merge_cells("A2:E2")
    ws["A2"] = "Saturdays and Sundays excluded"
    _style(ws["A2"], bg="EBF5FB",
           font=_font(bold=False, color="444444", size=10, italic=True),
           align=_center())
    ws.row_dimensions[2].height = 18

    # Headers
    headers = ['Date', 'Day', 'Media played', 'Printables used', 'Widgets Used']
    for col_idx, h in enumerate(headers, 1):
        c = ws.cell(3, col_idx, h)
        _style(c, bg=ACC, font=_font(size=10), align=_center(), border=_thin())
    ws.row_dimensions[3].height = 26

    # Data rows
    if daily_df is None or daily_df.empty:
        ws.merge_cells("A4:E4")
        ws["A4"] = "No DashBoard data available."
        ws["A4"].font = Font(name="Calibri", italic=True, color="888888")
        return
    for r_idx, (_, row) in enumerate(daily_df.iterrows()):
        er = r_idx + 4
        bg = ALT if r_idx % 2 == 0 else "FFFFFF"
        ws.row_dimensions[er].height = 20

        vals = [row['Date'], row['Day'],
                row['Media played'], row['Printables used'], row['Widgets Used']]
        for col_idx, val in enumerate(vals, 1):
            c = ws.cell(er, col_idx, val)
            _style(c, bg=bg,
                   font=_font(bold=False, color="222222", size=10),
                   align=_left() if col_idx <= 2 else _center(),
                   border=_thin())

    # Totals row
    tot_row = len(daily_df) + 4
    ws.row_dimensions[tot_row].height = 24
    total_media  = int(daily_df['Media played'].sum())
    total_prints = int(daily_df['Printables used'].sum())
    total_widgets= int(daily_df['Widgets Used'].sum())

    for col_idx, val in enumerate(['TOTAL', '', total_media, total_prints, total_widgets], 1):
        c = ws.cell(tot_row, col_idx, val)
        _style(c, bg=HDR, font=_font(bold=True, color="FFFFFF", size=11),
               align=_left() if col_idx == 1 else _center(),
               border=_med())

    print(f"     • Daily Usage sheet added ({len(daily_df)} rows)")
    print(f"       Totals → Media: {total_media}  Printables: {total_prints}  Widgets: {total_widgets}")



# ─────────────────────────────────────────────
# TEACHER SUMMARY SHEET
# ─────────────────────────────────────────────
TEACHER_RECOMMENDED_MINS = 450   # Recommended usage mins for all teachers

def build_sheet_teacher_summary(wb, xl):
    """
    New sheet: Teacher Summary
    Columns: Name of the Teacher | Subject Taught |
             Recommended Usage (mins) | Total Session Duration (mins)
    - Removes Support Admin
    - Recommended Usage = 450 mins for all teachers
    - Colour codes Total Session: green if >= 450, orange if < 450
    - Sorted by Session Duration descending
    """
    import numpy as np
    df = xl.parse('UserSummary')
    df = df[~df['Name'].str.contains('Support Admin', case=False, na=False)].reset_index(drop=True)
    df = df.sort_values('Session Duration (mins)', ascending=False).reset_index(drop=True)

    ws = wb.create_sheet("Teacher Summary")
    ws.sheet_view.showGridLines = False

    # Column widths
    ws.column_dimensions['A'].width = 32
    ws.column_dimensions['B'].width = 48
    ws.column_dimensions['C'].width = 24
    ws.column_dimensions['D'].width = 28

    # Title
    ws.merge_cells("A1:D1")
    ws["A1"] = "Teacher Usage Summary"
    _style(ws["A1"], bg=HDR, font=_font(size=13), align=_center())
    ws.row_dimensions[1].height = 34

    # Sub-info
    ws.merge_cells("A2:D2")
    ws["A2"] = f"Recommended Usage: {TEACHER_RECOMMENDED_MINS} mins   |   Total Teachers: {len(df)}"
    _style(ws["A2"], bg="EBF5FB",
           font=_font(bold=False, color="444444", size=10, italic=True),
           align=_center())
    ws.row_dimensions[2].height = 18

    # Headers
    headers = ['Name of the Teacher', 'Subject Taught',
               'Recommended Usage (mins)', 'Total Session Duration (mins)']
    for col_idx, h in enumerate(headers, 1):
        c = ws.cell(3, col_idx, h)
        _style(c, bg=ACC, font=_font(size=10), align=_center(), border=_thin())
    ws.row_dimensions[3].height = 28

    # Data rows
    for r_idx, (_, row) in enumerate(df.iterrows()):
        er  = r_idx + 4
        bg  = ALT if r_idx % 2 == 0 else "FFFFFF"
        dur = int(row['Session Duration (mins)']) if not (isinstance(row['Session Duration (mins)'], float) and np.isnan(row['Session Duration (mins)'])) else 0
        meets = dur >= TEACHER_RECOMMENDED_MINS
        ws.row_dimensions[er].height = 20

        # Col A: Name
        c1 = ws.cell(er, 1, row['Name'])
        _style(c1, bg=bg, font=_font(bold=False, color="222222", size=10),
               align=_left(), border=_thin())

        # Col B: Subject Taught
        subj = str(row['Subjects taught']) if pd.notna(row['Subjects taught']) else '—'
        c2 = ws.cell(er, 2, subj)
        _style(c2, bg=bg, font=_font(bold=False, color="222222", size=10),
               align=_left(), border=_thin())
        c2.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)

        # Col C: Recommended Usage (mins)
        c3 = ws.cell(er, 3, TEACHER_RECOMMENDED_MINS)
        _style(c3, bg=bg, font=_font(bold=False, color="555555", size=10, italic=True),
               align=_center(), border=_thin())

        # Col D: Total Session Duration — green if meets, orange if not
        c4 = ws.cell(er, 4, dur)
        _style(c4, bg=bg,
               font=_font(bold=True, color=GRN if meets else ORG, size=10),
               align=_center(), border=_thin())

    # Totals row
    tot_row = len(df) + 4
    ws.row_dimensions[tot_row].height = 24
    total_dur = int(df['Session Duration (mins)'].fillna(0).sum())
    meets_count = int((df['Session Duration (mins)'].fillna(0) >= TEACHER_RECOMMENDED_MINS).sum())

    for col_idx, val in enumerate(
        [f'TOTAL  ({meets_count} teachers met >= {TEACHER_RECOMMENDED_MINS} mins)',
         '', '', total_dur], 1):
        c = ws.cell(tot_row, col_idx, val)
        _style(c, bg=HDR, font=_font(bold=True, color="FFFFFF", size=11),
               align=_left() if col_idx == 1 else _center(), border=_med())

    print(f"     • Teacher Summary sheet: {len(df)} teachers  |  {meets_count} met >= {TEACHER_RECOMMENDED_MINS} mins")


def build_excel(t_df, t_met, c_df, c_met, det_df, class_tables,
                working_days, out_folder, daily_df=None, xl=None):
    print("\n📊 Building Excel dashboard…")
    wb = Workbook()

    # ── Sheet 1: Summary ─────────────────────────────────────
    ws = wb.active; ws.title = "Summary Dashboard"
    ws.sheet_view.showGridLines = False
    for col, w in zip('ABCD', [42,20,20,20]):
        ws.column_dimensions[col].width = w
    _title(ws, "📊 School Usage Dashboard", 4)
    ws.merge_cells("A2:D2")
    ws["A2"] = f"Generated: {datetime.now().strftime('%d %b %Y  %H:%M')}   |   Working Days: {working_days}"
    _style(ws["A2"], bg="EBF5FB",
           font=_font(bold=False, color="444444", size=10, italic=True),
           align=_center())
    ws.row_dimensions[2].height = 18

    sections = [
        ("👩‍🏫  TEACHER METRICS",                    None,                None, HDR,  True),
        ("Total Teachers (excl. Admin)",             t_met['total'],      "#",  ACC,  False),
        ("Eligible Teachers",                        t_met['eligible'],   "#",  ACC,  False),
        ("Teachers with total session > 30 mins",    t_met['above_30'],   "#",  GRN,  False),
        ("% Teachers > 30 mins",                     t_met['pct_above_30'],"%", GRN,  False),
        ("Teachers averaging ≥ 30 mins/day",         t_met['avg_30'],     "#",  ORG,  False),
        ("% averaging ≥ 30 mins/day (of eligible)",  t_met['pct_avg_30'], "%",  ORG,  False),
        ("🏫  CLASSROOM METRICS",                    None,                None, HDR,  True),
        ("Total Classrooms",                         c_met['total'],      "#",  ACC,  False),
        ("Classrooms with Duration > 30 mins",       c_met['above_30'],   "#",  GRN,  False),
        ("% Classrooms > 30 mins",                   c_met['pct_above_30'],"%", GRN,  False),
        ("Classrooms averaging ≥ 40 mins/day",       c_met['above_40'],   "#",  ORG,  False),
        ("% Classrooms avg ≥ 40 mins/day",           c_met['pct_above_40'],"%", ORG,  False),
    ]
    for i, (label, val, fmt, color, is_hdr) in enumerate(sections, start=4):
        ws.row_dimensions[i].height = 28 if is_hdr else 24
        bg = HDR if is_hdr else ("F7FAFD" if i%2==0 else "FFFFFF")
        if is_hdr:
            ws.merge_cells(f"A{i}:D{i}")
            c = ws.cell(i, 1, label)
            _style(c, bg=HDR, font=_font(size=11), align=_center(), border=_thin())
        else:
            c1 = ws.cell(i, 1, label)
            _style(c1, bg=bg, font=_font(bold=False, color="222222", size=10),
                   align=_left(), border=_thin())
            disp = f"{val}%" if fmt=="%" else val
            c2 = ws.cell(i, 2, disp)
            _style(c2, bg=bg, font=_font(bold=True, color=color, size=13),
                   align=_center(), border=_thin())
            ws.merge_cells(f"C{i}:D{i}")
            for col in (3,4):
                cx = ws.cell(i, col)
                _style(cx, bg=bg, border=_thin())
    print("   • Sheet 1 / Summary Dashboard  ✅")

    # ── Sheet 2: Teacher Detail ───────────────────────────────
    ws2 = wb.create_sheet("Teacher Detail")
    ws2.sheet_view.showGridLines = False
    _title(ws2, "Teacher Session Detail", 6)
    _hdr_row(ws2, 2,
             ['Name','Session Duration (mins)','Avg Daily (mins)',
              'Days Accessed','Media Played','Avg ≥ 30 mins/day'],
             [30,24,20,16,16,18])
    sdf = t_df.sort_values('Session Duration (mins)', ascending=False).reset_index(drop=True)
    for r, row in sdf.iterrows():
        er   = r+3; bg = ALT if r%2==0 else "FFFFFF"
        avg  = round(row.get('Avg Daily (mins)',
                             row['Session Duration (mins)']/working_days), 1)
        flag = "✅ Yes" if avg>=30 else "❌ No"
        _row(ws2, er,
             [row['Name'], row['Session Duration (mins)'], avg,
              row.get('Days accessed',''), row.get('Media played',''), flag],
             bg, lefts=(1,))
        ws2.cell(er,6).font = _font(bold=True,
                                    color=GRN if flag.startswith("✅") else ORG, size=10)
    print("   • Sheet 2 / Teacher Detail      ✅")

    # ── Sheet 3: Classroom Detail ─────────────────────────────
    ws3 = wb.create_sheet("Classroom Detail")
    ws3.sheet_view.showGridLines = False
    _title(ws3, "Classroom Usage Detail", 5)
    _hdr_row(ws3, 2,
             ['Classroom','Total Duration (mins)','Avg Daily (mins)',
              'Days Accessed','Avg ≥ 40 mins/day'],
             [32,22,20,16,18])
    cdf = c_df.sort_values('Duration(mins)', ascending=False).reset_index(drop=True)
    for r, row in cdf.iterrows():
        er   = r+3; bg = ALT if r%2==0 else "FFFFFF"
        avg  = round(row['Duration(mins)']/working_days, 1)
        flag = "✅ Yes" if avg>=40 else "❌ No"
        _row(ws3, er,
             [row['Classroom accessed'], int(row['Duration(mins)']),
              avg, int(row.get('Days accessed',0)), flag],
             bg, lefts=(1,))
        ws3.cell(er,5).font = _font(bold=True,
                                    color=GRN if flag.startswith("✅") else ORG, size=10)
    print("   • Sheet 3 / Classroom Detail    ✅")

    # ── Sheet 4: Detailed Usage ───────────────────────────────
    ws4 = wb.create_sheet("Detailed Usage")
    ws4.sheet_view.showGridLines = False
    _title(ws4, "Detailed Usage – Cleaned & Formatted", 7)
    ws4.merge_cells("A2:G2")
    ws4["A2"] = (f"Sessions: {len(det_df)}   |   "
                 f"With duration: {int(det_df['Duration (mins)'].notna().sum())}   |   "
                 f"Avg: {round(det_df['Duration (mins)'].mean(),1)} mins")
    _style(ws4["A2"], bg="EBF5FB",
           font=_font(bold=False, color="444444", size=10, italic=True),
           align=_center())
    ws4.row_dimensions[2].height = 18
    _hdr_row(ws4, 3,
             ['Name','Date','startTime','endTime',
              'Duration (mins)','subjectTitle','gradeTitle'],
             [28,14,22,22,18,28,18])
    sdet = det_df.sort_values(['Name','Date','startTime']).reset_index(drop=True)
    for r, row in sdet.head(5000).iterrows():
        er = r+4; bg = ALT if r%2==0 else "FFFFFF"
        dur = row.get('Duration (mins)','')
        _row(ws4, er,
             [row.get('Name',''), str(row.get('Date',''))[:10],
              str(row.get('startTime',''))[:19] if pd.notna(row.get('startTime')) else '',
              str(row.get('endTime',''))[:19]   if pd.notna(row.get('endTime'))   else '',
              dur, row.get('subjectTitle',''), row.get('gradeTitle','')],
             bg, lefts=(1,6,7))
        ws4.row_dimensions[er].height = 18
        if isinstance(dur, float) and not pd.isna(dur):
            ws4.cell(er,5).font = _font(bold=True,
                                        color=GRN if dur>=30 else ORG, size=10)
    print("   • Sheet 4 / Detailed Usage      ✅")

    # ── Sheet 5: Subject-wise Grade Usage ───────────────────
    def _subj_color(subject):
        if subject in {'English'}:                              return ('1A5FBA','FFFFFF')
        if subject in {'Mathematics'}:                          return ('7C3AED','FFFFFF')
        if subject in {'Hindi'}:                                return ('DC2626','FFFFFF')
        if subject in {'Science','Biology','Chemistry','Physics',
                       'Computer Science','Artificial Intelligence',
                       'Psychology','Science Practicals'}:      return ('0D9488','FFFFFF')
        if subject in {'Social Studies','Geography','History',
                       'Political Science','Economics',
                       'Business Studies'}:                     return ('D97706','FFFFFF')
        return ('475569','FFFFFF')

    ws5 = wb.create_sheet("Subject x Grade Usage")
    ws5.sheet_view.showGridLines = False
    ws5.column_dimensions['A'].width = 28
    ws5.column_dimensions['B'].width = 26
    ws5.column_dimensions['C'].width = 18
    _title(ws5, "Subject-wise Usage by Grade", 2)
    row_ptr = 3
    for subject, tbl in class_tables.items():
        if tbl.empty: continue
        bg_hex, fg_hex = _subj_color(subject)
        total = round(tbl['Usage (mins)'].sum(), 2)
        ws5.merge_cells(f"A{row_ptr}:B{row_ptr}")
        ws5[f"A{row_ptr}"] = f"{subject}   |   Total: {total} mins"
        _style(ws5[f"A{row_ptr}"], bg=bg_hex,
               font=_font(size=12, color=fg_hex),
               align=_left(), border=_thin())
        ws5.row_dimensions[row_ptr].height = 26
        row_ptr += 1
        _hdr_row(ws5, row_ptr,
                 ['gradeTitle','Recommended Usage (mins)','Usage (mins)'],
                 [28,26,18], height=22)
        row_ptr += 1
        for ri, (_, dr) in enumerate(tbl.iterrows()):
            bg   = ALT if ri%2==0 else "FFFFFF"
            uval = dr['Usage (mins)']
            c1 = ws5.cell(row_ptr,1, dr['Grade Title'])
            _style(c1, bg=bg, font=_font(bold=False,color="222222",size=10),
                   align=_left(), border=_thin())
            rec  = dr.get('Recommended (mins)', '—')
            meets = isinstance(rec, (int, float)) and uval >= rec

            c2 = ws5.cell(row_ptr, 2, int(rec) if isinstance(rec,(int,float)) else '—')
            _style(c2, bg=bg, font=_font(bold=False,color="555555",size=10,italic=True),
                   align=_center(), border=_thin())

            c3 = ws5.cell(row_ptr, 3, uval)
            _style(c3, bg=bg,
                   font=_font(bold=True, color=GRN if meets else ORG, size=10),
                   align=_center(), border=_thin())
            ws5.row_dimensions[row_ptr].height = 20
            row_ptr += 1
        for col, val in enumerate([f"TOTAL – {subject}", '', total],1):
            c = ws5.cell(row_ptr,col,val)
            _style(c, bg=bg_hex,
                   font=_font(bold=True,color=fg_hex,size=11),
                   align=_left() if col==1 else _center(),
                   border=_med())
        ws5.row_dimensions[row_ptr].height = 22
        row_ptr += 3
    print("   • Sheet 5 / Subject x Grade     ✅")
    build_sheet_daily(wb, daily_df)
    print("   • Sheet 6 / Daily Usage         ✅")
    build_sheet_teacher_summary(wb, xl)
    print("   • Sheet 7 / Teacher Summary     ✅")

    ts    = datetime.now().strftime('%Y%m%d_%H%M')
    fpath = os.path.join(out_folder, f"dashboard_{ts}.xlsx")
    wb.save(fpath)
    print(f"\n   💾 Excel saved → {fpath}")
    return fpath

# ================================================================
#  HTML BUILDER
# ================================================================

def build_html(t_met, c_met, det_df, class_tables, working_days, out_folder):
    print("\n🌐 Building HTML dashboard…")

    def subj_color_html(subject):
        if subject in {'English'}:                           return ('#1A5FBA','#FFFFFF')
        if subject in {'Mathematics'}:                       return ('#7C3AED','#FFFFFF')
        if subject in {'Hindi'}:                             return ('#DC2626','#FFFFFF')
        if subject in {'Science','Biology','Chemistry','Physics',
                       'Computer Science','Artificial Intelligence',
                       'Psychology','Science Practicals'}:   return ('#0D9488','#FFFFFF')
        if subject in {'Social Studies','Geography','History',
                       'Political Science','Economics',
                       'Business Studies'}:                  return ('#D97706','#FFFFFF')
        return ('#475569','#FFFFFF')

    class_html = ""
    for subject, tbl in class_tables.items():
        if tbl.empty: continue
        bg, fg   = subj_color_html(subject)
        total    = round(tbl['Usage (mins)'].sum(), 2)
        tid      = "t_" + re.sub(r'\W+','_', subject)
        sid      = "s_" + re.sub(r'\W+','_', subject)
        rows = ""
        for _, row in tbl.iterrows():
            uval = row['Usage (mins)']
            rows += (f"<tr>"
                     f"<td>{row['Grade Title']}</td>"
                     f"<td style='text-align:center;color:#1A5FBA;font-weight:700'>{uval}</td>"
                     f"</tr>")
        rows += (f"<tr style='background:{bg};color:{fg};font-weight:700'>"
                 f"<td>TOTAL</td>"
                 f"<td style='text-align:center'>{total}</td></tr>")
        class_html += f"""
<div class="cls-card">
  <div class="cls-hdr" style="background:{bg};color:{fg}">{subject} &nbsp;·&nbsp; {total} mins</div>
  <div class="cls-sr"><input class="srch" id="{sid}" type="text" placeholder="🔍 Search grade…"
    oninput="ft('{sid}','{tid}')"/></div>
  <table id="{tid}">
    <thead><tr><th>gradeTitle</th>
      <th style="text-align:center">Usage (mins)</th></tr></thead>
    <tbody>{rows}</tbody>
  </table>
</div>"""

    t = t_met; c = c_met
    html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>School Usage Dashboard</title>
<style>
:root{{--navy:#0F2744;--blue:#1A5FBA;--green:#16A34A;--orange:#EA580C;
  --bg:#F0F4FA;--card:#fff;--border:#DDE3EE;--text:#1E293B;--muted:#64748B}}
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:system-ui,sans-serif;background:var(--bg);color:var(--text)}}
header{{background:linear-gradient(135deg,var(--navy),var(--blue));padding:24px 40px;color:#fff}}
header h1{{font-size:22px;font-weight:700}}
header p{{font-size:13px;opacity:.7;margin-top:4px}}
.wrap{{max-width:1280px;margin:0 auto;padding:0 28px 60px}}
.sec{{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;
  color:var(--muted);margin:32px 0 14px}}
.kpi-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px}}
.kpi{{background:var(--card);border-radius:10px;padding:18px 20px;
  box-shadow:0 2px 12px rgba(15,39,68,.07);border:1px solid var(--border);
  border-left:4px solid var(--blue)}}
.kpi.g{{border-left-color:var(--green)}}.kpi.o{{border-left-color:var(--orange)}}
.kl{{font-size:11px;color:var(--muted);text-transform:uppercase;letter-spacing:.4px}}
.kv{{font-size:28px;font-weight:700;margin-top:5px;color:var(--navy)}}
.kpi.g .kv{{color:var(--green)}}.kpi.o .kv{{color:var(--orange)}}
.cls-grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:16px}}
.cls-card{{background:var(--card);border-radius:10px;overflow:hidden;
  box-shadow:0 2px 12px rgba(15,39,68,.07);border:1px solid var(--border)}}
.cls-hdr{{padding:11px 16px;font-weight:700;font-size:14px}}
.cls-sr{{padding:8px 12px;background:#FAFBFC;border-bottom:1px solid var(--border)}}
.srch{{width:100%;padding:6px 10px;border:1.5px solid var(--border);
  border-radius:7px;font-family:inherit;font-size:12px}}
table{{width:100%;border-collapse:collapse;font-size:13px}}
thead th{{background:#F8FAFC;padding:8px 12px;text-align:left;font-size:11px;
  font-weight:700;text-transform:uppercase;letter-spacing:.4px;
  color:var(--muted);border-bottom:1px solid var(--border)}}
tbody td{{padding:8px 12px;border-bottom:1px solid #F1F5F9}}
tbody tr:last-child td{{border-bottom:none}}
footer{{text-align:center;padding:20px;font-size:12px;color:var(--muted);
  border-top:1px solid var(--border);margin-top:20px}}
</style></head><body>
<header>
  <h1>📊 School Usage Dashboard</h1>
  <p>Generated: {datetime.now().strftime('%d %b %Y  %H:%M')} &nbsp;|&nbsp; Working Days: {working_days}</p>
</header>
<div class="wrap">
  <div class="sec">👩‍🏫 Teacher Metrics</div>
  <div class="kpi-grid">
    <div class="kpi"><div class="kl">Total Teachers</div><div class="kv">{t['total']}</div></div>
    <div class="kpi"><div class="kl">Eligible Teachers</div><div class="kv">{t['eligible']}</div></div>
    <div class="kpi g"><div class="kl">Teachers &gt; 30 mins</div><div class="kv">{t['above_30']}</div></div>
    <div class="kpi g"><div class="kl">% Teachers &gt; 30 mins</div><div class="kv">{t['pct_above_30']}%</div></div>
    <div class="kpi o"><div class="kl">Avg ≥ 30 mins/day</div><div class="kv">{t['avg_30']}</div></div>
    <div class="kpi o"><div class="kl">% Avg ≥ 30 mins</div><div class="kv">{t['pct_avg_30']}%</div></div>
  </div>
  <div class="sec">🏫 Classroom Metrics</div>
  <div class="kpi-grid">
    <div class="kpi"><div class="kl">Total Classrooms</div><div class="kv">{c['total']}</div></div>
    <div class="kpi g"><div class="kl">Classrooms &gt; 30 mins</div><div class="kv">{c['above_30']}</div></div>
    <div class="kpi g"><div class="kl">% Classrooms &gt; 30 mins</div><div class="kv">{c['pct_above_30']}%</div></div>
    <div class="kpi o"><div class="kl">Avg ≥ 40 mins/day</div><div class="kv">{c['above_40']}</div></div>
    <div class="kpi o"><div class="kl">% Avg ≥ 40 mins/day</div><div class="kv">{c['pct_above_40']}%</div></div>
  </div>
  <div class="sec">📚 Subject-wise Usage by Class</div>
  <div class="cls-grid">{class_html}</div>
</div>
<footer>Auto-generated · {len(det_df)} sessions processed</footer>
<script>
function ft(s,t){{const q=document.getElementById(s).value.toLowerCase();
  document.querySelectorAll('#'+t+' tbody tr:not(:last-child)').forEach(r=>{{
    r.style.display=r.cells[0]?.textContent.toLowerCase().includes(q)?'':'none'}})}}
</script></body></html>"""

    ts    = datetime.now().strftime('%Y%m%d_%H%M')
    fpath = os.path.join(out_folder, f"dashboard_{ts}.html")
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"   💾 HTML  saved → {fpath}")
    return fpath

# ================================================================
#  MAIN — UPLOAD + PROCESS + DOWNLOAD
# ================================================================

print()
print("=" * 60)
print("  SCHOOL USAGE DASHBOARD GENERATOR")
print("=" * 60)

# ── Upload file ──────────────────────────────────────────────
print()
print("STEP 1 ── Upload your Excel report")
print("Click the 'Choose Files' button that appears below:")
print()

from google.colab import files as _cf
_uploaded = _cf.upload()

if not _uploaded:
    raise SystemExit("No file uploaded. Please run the cell again.")

_fname = list(_uploaded.keys())[0]
_fbytes = _uploaded[_fname]

if not _fname.lower().endswith((".xlsx", ".xls")):
    raise ValueError(f"Please upload an .xlsx file. Got: {_fname}")

UPLOAD_PATH = f"/content/{_fname}"
with open(UPLOAD_PATH, "wb") as _f:
    _f.write(_fbytes if isinstance(_fbytes, bytes) else _fbytes.read())
print(f"\n✅ Uploaded: {_fname}  ({os.path.getsize(UPLOAD_PATH)/1024:.1f} KB)")

# ── Parameters ───────────────────────────────────────────────
print()
print("STEP 2 ── Enter parameters")
print("-" * 40)

while True:
    try:
        wd = input("Working days this month [default 15]: ").strip()
        WORKING_DAYS = int(wd) if wd else 15
        if WORKING_DAYS < 1: raise ValueError
        break
    except ValueError:
        print("Please enter a positive number.")

OUTPUT_FOLDER = "/content"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ── Process ──────────────────────────────────────────────────
print()
print("STEP 3 ── Processing data")
print("-" * 40)

xl          = pd.ExcelFile(UPLOAD_PATH)
missing     = [s for s in ['UserSummary','TerminalSummary','DetailedUsage']
               if s not in xl.sheet_names]
if missing:
    raise ValueError(f"Missing sheets: {missing}\nFound: {xl.sheet_names}")

t_df, t_met  = process_user_summary(xl, WORKING_DAYS)
c_df, c_met  = process_terminal_summary(xl, WORKING_DAYS)
det_df        = process_detailed_usage(xl)
class_tables  = build_class_tables(det_df)
daily_df      = process_dashboard_daily(xl)

# ── Build outputs ────────────────────────────────────────────
print()
print("STEP 4 ── Building output files")
print("-" * 40)

excel_path = build_excel(t_df, t_met, c_df, c_met,
                          det_df, class_tables,
                          WORKING_DAYS, OUTPUT_FOLDER, daily_df, xl)
html_path  = build_html(t_met, c_met, det_df, class_tables,
                         WORKING_DAYS, OUTPUT_FOLDER)

# ── Summary ──────────────────────────────────────────────────
print()
print("=" * 60)
print("  ✅  ALL DONE!")
print("=" * 60)
print(f"  📊 Excel  → {os.path.basename(excel_path)}")
print(f"  🌐 HTML   → {os.path.basename(html_path)}")
print("=" * 60)

# ── Auto-download ────────────────────────────────────────────
print()
print("📥 Downloading files to your computer…")
_cf.download(excel_path)
_cf.download(html_path)
print("✅ Download complete!")

✅ Libraries loaded

  SCHOOL USAGE DASHBOARD GENERATOR

STEP 1 ── Upload your Excel report
Click the 'Choose Files' button that appears below:



KeyboardInterrupt: 